# Preprocessing

Steps:
- [Data Cleaning](#1)
- [Outlier Handling](#2)
- [Feature Encoding](#3)
- [Feature Scaling](#4)

## Import Libraries and Load Data

In [141]:
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
import pandas as pd

### Load and Inspect Data

In [142]:
df = pd.read_csv('../data/raw/dataset.csv')  

# check data types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         1000 non-null   int64  
 1   No_Pation  1000 non-null   int64  
 2   Gender     1000 non-null   str    
 3   AGE        1000 non-null   int64  
 4   Urea       1000 non-null   float64
 5   Cr         1000 non-null   int64  
 6   HbA1c      1000 non-null   float64
 7   Chol       1000 non-null   float64
 8   TG         1000 non-null   float64
 9   HDL        1000 non-null   float64
 10  LDL        1000 non-null   float64
 11  VLDL       1000 non-null   float64
 12  BMI        1000 non-null   float64
 13  CLASS      1000 non-null   str    
dtypes: float64(8), int64(4), str(2)
memory usage: 109.5 KB


In [143]:
df.nunique()

ID           800
No_Pation    961
Gender         3
AGE           50
Urea         110
Cr           113
HbA1c        111
Chol          77
TG            69
HDL           48
LDL           65
VLDL          60
BMI           64
CLASS          5
dtype: int64

Note: The unique values for Gender and Class are 3 and 5 which is unusual so we further look into their values

In [144]:
print(df['CLASS'].unique())
print(df['Gender'].unique())

<StringArray>
['N', 'N ', 'P', 'Y', 'Y ']
Length: 5, dtype: str
<StringArray>
['F', 'M', 'f']
Length: 3, dtype: str


<a id='1'></a>
### Data Cleaning

String Trimming and Case Consistency

In [145]:
df['CLASS'] = df['CLASS'].str.strip()
df['Gender'] = df['Gender'].str.upper()

print(df['CLASS'].unique())
print(df['Gender'].unique())

<StringArray>
['N', 'P', 'Y']
Length: 3, dtype: str
<StringArray>
['F', 'M']
Length: 2, dtype: str


Drop Redundant Columns: We drop ID and No_Pation Column as they as no predictive power

In [146]:
df.drop(columns=['ID','No_Pation'], inplace=True)
df.head(2)

,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
0,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,M,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N


<a id='2'></a>
### Outlier Handling

In [147]:
num_cols = ['Urea','Cr','HbA1c','Chol','TG','HDL','LDL','VLDL','BMI','AGE']

for col in num_cols:
    lower_bound = df[col].quantile(0.05)
    upper_bound = df[col].quantile(0.95)

    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)


<a id='3'></a>
### Feature Encoding

Encode categorical variables (Gender → 0/1, CLASS → 0/1/2)

In [148]:
le_gender = LabelEncoder()
le_class  = LabelEncoder()
df['Gender_enc'] = le_gender.fit_transform(df['Gender'])   # F=0, M=1
df['CLASS_enc']  = le_class.fit_transform(df['CLASS'])     # N=0, P=1, Y=2

feature_cols = ['AGE','Urea','Cr','HbA1c','Chol','TG','HDL','LDL','VLDL','BMI','Gender_enc']
X = df[feature_cols].values
y = df['CLASS_enc'].values

print(f'Class mapping: {dict(zip(le_class.classes_, le_class.transform(le_class.classes_)))}')
print(f'Gender mapping: {dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))}')

Class mapping: {'N': np.int64(0), 'P': np.int64(1), 'Y': np.int64(2)}
Gender mapping: {'F': np.int64(0), 'M': np.int64(1)}


### <a id='4'></a>
### Feature Scaling
Standardize features (mean=0, std=1) — required for SVM, LR, KNN, clustering

In [149]:
# Define which columns to scale
num_features = ['AGE', 'Urea', 'Cr', 'HbA1c', 'Chol', 'TG', 'HDL', 'LDL', 'VLDL', 'BMI']
passthrough_features = ['Gender_enc']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('pass', 'passthrough', passthrough_features)
    ])

# X should NOT include 'CLASS' or 'CLASS_enc'
X = df[num_features + passthrough_features]
X_scaled = preprocessor.fit_transform(X)

Saving processed data

In [150]:
# save processed data
processed_df = pd.DataFrame(X_scaled, columns=feature_cols)
processed_df['CLASS_enc'] = y
processed_df.to_csv('../data/processed/diabetes_processed.csv', index=False)

In [152]:
processed_df = pd.read_csv('../data/processed/diabetes_processed.csv')
print(processed_df.head())


        AGE      Urea        Cr     HbA1c      Chol        TG       HDL  \
0 -0.428865 -0.107732 -0.837168 -1.425119 -0.587854 -1.206459  2.220443   
1 -2.552580 -0.218004 -0.032658 -1.425119 -1.045029 -0.773325 -0.143459   
2 -0.428865 -0.107732 -0.837168 -1.425119 -0.587854 -1.206459  2.220443   
3 -0.428865 -0.107732 -0.837168 -1.425119 -0.587854 -1.206459  2.220443   
4 -2.552580  1.215531 -0.837168 -1.425119  0.052191 -1.119832 -1.029923   

        LDL      VLDL       BMI  Gender_enc  CLASS_enc  
0 -1.226176 -0.562434 -1.164901         0.0          0  
1 -0.490227 -0.504301 -1.375503         1.0          0  
2 -1.226176 -0.562434 -1.164901         0.0          0  
3 -1.226176 -0.562434 -1.164901         0.0          0  
4 -0.595362 -0.620567 -1.796708         1.0          0  
